## Aplicación de genAI para asesoría legal

En este proyecto se quiere hacer uso de LLMs para que usuarios, con un lenguaje coloquial y sin conocimientos en el área legal, puedan consultar y asesorarse de sobre demandas asociadas a redes sociales.

Esta prueba de concepto hace uso de la vectorizacion de texto a traves de librerias como `langchain-openai` para que estos puedan ser almacenados con mayor robustez en una base de datos vectorial, y consultado por un manejador de este tipo de repositorios, como lo es `chromaDB` o `faiss`.

### 1. Lectura de datos

In [21]:
# Essential.
import pandas as pd
import os
from dotenv import load_dotenv

# RAG.
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter as RCT
from langchain_community.vectorstores import Chroma
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

#### 1.1 Describiendo el dataset.

In [22]:
raw_data = pd.read_excel('./data/sentencias_pasadas.xlsx')

In [23]:
raw_data.isna().sum() # Los datos poseen varios valores faltantes.

#                    0
Relevancia           0
Providencia          0
Tipo               329
Fecha Sentencia      0
Tema - subtema      92
resuelve             0
sintesis             0
dtype: int64

In [24]:
raw_data[raw_data["Tipo"].isna()].loc[:, ["Tipo"]]

,Tipo
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
...,...
324,NaN
325,NaN
326,NaN
327,NaN


In [25]:
raw_data.iloc[3:5, :]

,#,Relevancia,Providencia,Tipo,Fecha Sentencia,Tema - subtema,resuelve,sintesis
3,6,955889.0,T-246/21,NaN,2021-07-29,ACCION DE TUTELA PARA PROTEGER EL DERECHO A LA...,en nombre del pueblo y por mandato de la Const...,Se presenta la acción de tutela en contra de u...
4,7,955787.0,T-245A/22,NaN,2022-07-01,ACCION DE TUTELA-Inexistencia de hecho superad...,en nombre del pueblo y por mandato de la Const...,"El accionante, actuando en representación de s..."


In [26]:
raw_data[raw_data["Tema - subtema"].isna()].loc[:, ["Tema - subtema"]]

,Tema - subtema
0,NaN
5,NaN
6,NaN
12,NaN
13,NaN
...,...
316,NaN
320,NaN
321,NaN
326,NaN


In [27]:
raw_data.iloc[4:7, :]

,#,Relevancia,Providencia,Tipo,Fecha Sentencia,Tema - subtema,resuelve,sintesis
4,7,955787.0,T-245A/22,NaN,2022-07-01,ACCION DE TUTELA-Inexistencia de hecho superad...,en nombre del pueblo y por mandato de la Const...,"El accionante, actuando en representación de s..."
5,8,954029.0,T-190/24,NaN,2024-05-23,NaN,RESUELVE PRIMERO. CONFIRMAR la decisión profer...,El actor solicitó la protección de los derecho...
6,9,947406.0,T-394/24,NaN,2024-09-19,NaN,RESUELVE PRIMERO. CONFIRMAR la Sentencia del 1...,La presente acción de tutela fue formulada por...


Los valores faltantes se ven asociados a espacios en blanco, que no deben ser llenados con texto porque pueden introducir información errónea. Por lo tanto, se procede a llenar con un espacio en blanco.

In [28]:
data = raw_data.copy()

In [29]:
data[["Tipo", "Tema - subtema"]] = raw_data[["Tipo", "Tema - subtema"]].fillna("")

In [30]:
data.tail()

,#,Relevancia,Providencia,Tipo,Fecha Sentencia,Tema - subtema,resuelve,sintesis
324,487,106681.0,SU.016/21,,2021-01-21,ABUSO DEL DERECHO-Elementos que lo configuran ...,en nombre del pueblo y por mandato de la Const...,"En este caso, el accionante y 56 personas más,..."
325,489,100241.0,SU.257/21,,2021-08-05,ACCION DE TUTELA CONTRA PROVIDENCIAS JUDICIALE...,en nombre del pueblo y por mandato de la Const...,La acción de tutela se interpuso en contra de ...
326,491,99781.0,C-489/23,,2023-11-16,,RESUELVE ÚNICO. Declarar INEXEQUIBLE el parágr...,Demanda de inconstitucionalidad contra el artí...
327,493,84998.0,C-293/20,,2020-08-05,,"administrando justicia en nombre del Pueblo, y...",Revisión de constitucionalidad del Decreto Leg...
328,495,81298.0,C-294/21,,2021-09-02,"NIÑOS, NIÑAS Y ADOLESCENTES COMO SUJETOS DE ES...",RESUELVE: PRIMERO.- Declarar la INEXEQUIBILIDA...,Demandas de inconstitucionalidad contra el Act...


In [31]:
data.dtypes

#                           int64
Relevancia                float64
Providencia                   str
Tipo                       object
Fecha Sentencia    datetime64[us]
Tema - subtema                str
resuelve                      str
sintesis                      str
dtype: object

In [32]:
data.set_index("#", inplace=True)

#### 1.2 Chunking

Con el fin de transformar las cadenas de texto en vectores, es necesario entregar longitudes de texto más cortas para que el procesamiento por el modelo de embeddings sea más eficiente, asímismo como el del LLM que recibirá el prompt.

In [33]:
# Cadena más larga por cada columna de texto
for col_name, col_values in data[["Tema - subtema", "resuelve", "sintesis"]].items():
    print(f"Rango de longitud para cada columna: {col_name}: {(col_values.str.len().min() , col_values.str.len().max())}")

Rango de longitud para cada columna: Tema - subtema: (np.int64(0), np.int64(10837))
Rango de longitud para cada columna: resuelve: (np.int64(180), np.int64(32767))
Rango de longitud para cada columna: sintesis: (np.int64(298), np.int64(3632))


In [34]:
data["sintesis"].str.len().sort_values(ascending=False)


#
52     3632
484    3537
465    3298
489    3224
468    3221
       ... 
193     360
356     356
82      348
220     346
215     298
Name: sintesis, Length: 329, dtype: int64

In [35]:
texty = data["sintesis"].iloc[38] 
# El texto mas largo de todo el dataframe.

In [36]:
def create_chunks(text, chunk_size, overlap):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i : i + chunk_size]
        chunks.append(chunk)
    return chunks

In [37]:
create_chunks(texty, 800, 100)

['El actor es profesor de Planta del Departamento de Sociología de la Facultad de Ciencias Humanas de la Universidad Nacional de Bogotá, de la cual se ha desempeñado como decano y candidato a Rector. En su ejercicio académico se ha reconocido como un hombre gay miembro de la comunidad LGBTI, motivo por el cual, según sus manifestaciones ha sido víctima de ataques y discriminaciones en razón a sus preferencias sexuales. La accionada es, igualmente, académica y egresada del departamento de Antropología de la misma institución universitaria y se reconoce como mujer feminista y defensora de los derechos humanos de las mujeres, motivo por el cual, según su indicación, ha liderado investigaciones sociales dirigidas a documentar casos de agresiones de profesores contra estudiante en contextos acadé',
 'ciales dirigidas a documentar casos de agresiones de profesores contra estudiante en contextos académicos. En este caso el accionante aduce que los informes difundidos por la demandada, en los 

### 2. Creando la base de datos vectorial.

In [38]:
# Importando la API key de OpenAI.
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [ ]:
# Init cliente
dk_client = chromadb.PersistentClient(path="./vector_db")

In [40]:
langchain_embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002", # La misma que se encuentra en Azure
    api_key=OPENAI_API_KEY
)

In [41]:
# Para que chroma acepte el vectorizador de embeddings.
from chromadb import EmbeddingFunction, Documents, Embeddings

class LangChainEmbeddingAdapter(EmbeddingFunction):
    def __init__(self, openai_ef):
        self.openai_ef = openai_ef
    def __call__(self, input:Documents) -> Embeddings:
        return self.openai_ef.embed_documents(input)
    def name(self) -> str:
        return "openai"
    
# Adaptador
adapter = LangChainEmbeddingAdapter(langchain_embeddings)

In [42]:
# Nombre coleccion
deptName = "departamento_legal_poc"

In [43]:
collection = dk_client.get_or_create_collection(
    name=deptName,
    embedding_function=adapter
)

In [44]:
documents = []
metadata = []
ids = []

In [45]:
# Instancia de RecursiveSplitter
text_splitter = RCT(chunk_size=800, chunk_overlap=100)

In [46]:
for idx, row in data.iterrows():
    full_text = f"TEMA : {row['Tema - subtema']} \n SINTESIS : {row['sintesis']} \n RESUELVE : {row['resuelve']}"
    
    chunks = text_splitter.split_text(full_text)
    for i, chunk in enumerate(chunks):
        documents.append(chunk)
        metadata.append(
            {
                "tema" : str(row['Tema - subtema']),
                "row_index" : idx
            }
        )
        ids.append(f"doc_{idx}_chunk_{i}")

In [47]:
total_length_chars = [] # Aproximado de tokens
for line in documents:
    for c in line:
        total_length_chars.append(c)

In [48]:
print(f"Tokens aproximados de `documents`: {len(total_length_chars)}\nmayor que 300.000")

Tokens aproximados de `documents`: 1803069
mayor que 300.000


In [49]:
batch = 100

In [50]:
for i in range(0, len(documents), batch):
    
    # Indexing con batches.
    batch_ids = ids[i : i + batch]
    batch_docs = documents[i : i + batch]
    batch_metadatas = metadata[i : i + batch]
    
    collection.add(
        ids=batch_ids,
        documents=batch_docs,
        metadatas=batch_metadatas
    )
    print(f"Uploaded batch {i // batch + 1} of {len(documents) // batch + 1}")

Uploaded batch 1 of 28
Uploaded batch 2 of 28
Uploaded batch 3 of 28
Uploaded batch 4 of 28
Uploaded batch 5 of 28
Uploaded batch 6 of 28
Uploaded batch 7 of 28
Uploaded batch 8 of 28
Uploaded batch 9 of 28
Uploaded batch 10 of 28
Uploaded batch 11 of 28
Uploaded batch 12 of 28
Uploaded batch 13 of 28
Uploaded batch 14 of 28
Uploaded batch 15 of 28
Uploaded batch 16 of 28
Uploaded batch 17 of 28
Uploaded batch 18 of 28
Uploaded batch 19 of 28
Uploaded batch 20 of 28
Uploaded batch 21 of 28
Uploaded batch 22 of 28
Uploaded batch 23 of 28
Uploaded batch 24 of 28
Uploaded batch 25 of 28
Uploaded batch 26 of 28
Uploaded batch 27 of 28
Uploaded batch 28 of 28


In [51]:
db = Chroma(
    client=dk_client,
    collection_name=deptName,
    embedding_function=langchain_embeddings
)

/tmp/ipykernel_13300/303402196.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(


In [52]:
metadata_info = [
    AttributeInfo(
        name="tema",
        description="La sintesis sobre el tema de la(s) tutela(s) y proceso(s) legales llevados a cabo.",
        type="string"
    )
]

In [53]:
# Contexto del documento
document_content_description = "Información sobre posibles demandas y sus resultados, relacionadas mayoritariamente con redes sociales. "

### 3. Consultas sobre la información legal.

In [54]:
openai_llm_instance = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model="gpt-4o",
    temperature=0
)

In [55]:
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever

In [56]:
retriever = SelfQueryRetriever.from_llm(
    llm=openai_llm_instance,
    vectorstore=db,
    document_contents=document_content_description,
    metadata_field_info=metadata_info,
    verbose=True
)

In [57]:
question = "¿Cuál fue la sentencia del caso que habla de acoso escolar?"

In [58]:
retriever.invoke(question)

[Document(metadata={'tema': 'ACCION DE TUTELA CONTRA DECISIONES DISCIPLINARIAS-Criterios de procedencia DEBIDO PROCESO DISCIPLINARIO EN ESTABLECIMIENTO EDUCATIVO-Aspectos que se deben tener en cuenta en trámite sancionatorio DEBIDO PROCESO DISCIPLINARIO Y DERECHO A LA EDUCACIÓN-Vulneración en procedimiento administrativo sancionatorio del SENA, que determinó la cancelación de matrícula de la estudiante DEBIDO PROCESO EN ACTUACIONES DISCIPLINARIAS EN INSTITUCIONES EDUCATIVAS-Reiteración de jurisprudencia DERECHO A LA EDUCACION Y PERMANENCIA EN EL SISTEMA EDUCATIVO-Fundamental DERECHO A LA EDUCACION-Características HOSTIGAMIENTO O ACOSO DIGITAL EN ENTORNOS EDUCATIVOS (CIBERACOSO/CYBERBULLING)-Concepto HOSTIGAMIENTO O ACOSO ESCOLAR (BULLYING/MATONEO)-Caracterización RETOS EDUCATIVOS EN EL CONTEXTO DE TECNOLOGIAS DE LA INFORMACION Y LA COMUNICACION-Acoso escolar o matoneo en redes sociales', 'row_index': 56}, page_content='O ACOSO ESCOLAR (BULLYING/MATONEO)-Caracterización RETOS EDUCATIVOS

In [60]:
# To create a response.
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [61]:
system_prompt = (
    "You are a specialized legal assistant. Use the following pieces of retrieved "
    "context to answer the user's question accurately. "
    "If you don't know the answer, say that you don't know. "
    "Context: {context}"
)

In [63]:
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [64]:
question_answer_chain = create_stuff_documents_chain(openai_llm_instance, prompt)

In [65]:
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [66]:
def get_legal_response(user_query):
    # This invokes the retriever, then the LLM
    response = rag_chain.invoke({"input": user_query})
    
    # 4. Format the output as you requested
    # We extract the answer and unique metadata from the source documents
    formatted_response = {
        "output": response["answer"],
        "metadata": [doc.metadata for doc in response["context"]]
    }
    
    return formatted_response

# Example usage
final_result = get_legal_response("What happened in the Colegio XYZ case regarding bullying?")
print(final_result["output"])

In the Colegio XYZ case, the institution was found to have violated the rights to education, due process, privacy, dignity, and good name of a student due to their failure to investigate incidents of cyberbullying that occurred in the first semester of 2023. The lack of action on the part of the school led to a situation where the student was eventually withdrawn from the school where they were a victim of bullying.


In [59]:
# from langchain_openai import ChatOpenAI
# from langchain.chains import create_retrieval_chain
# from langchain.chains.combine_documents import create_stuff_documents_chain
# from langchain_core.prompts import ChatPromptTemplate

# # 1. Initialize the Chat Model (Modern replacement for OpenAI)
# llm = ChatOpenAI(
#     model="gpt-4o", 
#     temperature=0, 
#     openai_api_key=OPENAI_API_KEY
# )

# # 2. Define your Question
# question = "What are the legal precedents related to Civil Liability?"

# # 3. Direct Retrieval (Using the modern 'invoke' method)
# # This replaces get_relevant_documents
# docs = retriever.invoke(question)

# # 4. Construct the RAG Chain (Replacing RetrievalQA)
# # We create a prompt that tells the AI how to use the documents
# system_prompt = (
#     "You are a legal assistant. Use the following context to answer the question. "
#     "Context: {context}"
# )
# prompt = ChatPromptTemplate.from_messages([
#     ("system", system_prompt),
#     ("human", "{input}"),
# ])

# # Create the chain that combines docs and the retrieval logic
# question_answer_chain = create_stuff_documents_chain(llm, prompt)
# rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# # 5. Execute
# response = rag_chain.invoke({"input": question})

# print(response["answer"])